# Notebook 05 — GRU
**Role:** Deep learning baseline — lighter LSTM alternative  
**Data:** `Final_Cleaned_1m.parquet`  
**Features:** `total_traffic`, `avg_node_stress`, `replica_count`, `avg_response_time`  
**Target:** `total_cpu_demand`  
**Pipeline:** identical to LSTM — every-4th val split, log1p + MinMaxScaler  
**Metrics:** R², MAE, RMSE, MAPE → `results/all_models_results.csv`

In [1]:
import random, os, warnings
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PATH    = str(PROJECT_ROOT / 'cleanData' / 'Final_Cleaned_1m.parquet')
PLOTS_DIR    = str(PROJECT_ROOT / 'plots')
RESULTS_CSV  = str(PROJECT_ROOT / 'results' / 'all_models_results.csv')
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(str(PROJECT_ROOT / 'results'), exist_ok=True)

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
                     'axes.grid':True,'grid.alpha':0.3,'font.size':11})

def save_result(model_name, r2, mae, rmse, mape):
    row = pd.DataFrame([{'model':model_name,'R2':round(float(r2),4),
        'MAE':round(float(mae),4),'RMSE':round(float(rmse),4),'MAPE':round(float(mape),2)}])
    if os.path.exists(RESULTS_CSV):
        ex = pd.read_csv(RESULTS_CSV)
        row = pd.concat([ex[ex['model']!=model_name], row], ignore_index=True)
    row.to_csv(RESULTS_CSV, index=False)
    print(f"  Saved \u2192 results/all_models_results.csv")

print(f"Project root : {PROJECT_ROOT}")
print(f"Data         : {DATA_PATH}")


Project root : C:\Users\phone\Documents\Alibaba\Cloud_Autoscale
Data         : C:\Users\phone\Documents\Alibaba\Cloud_Autoscale\cleanData\Final_Cleaned_1m.parquet


## 1. Load Data

In [2]:
print("Loading Cleaned 1-Minute Data...")
df_final_1m = pl.read_parquet(DATA_PATH)
print(f"Loaded Shape: {df_final_1m.shape}")
print(f"Columns: {df_final_1m.columns}")
print(f"Timestamp range: {df_final_1m['timestamp'].min()} to {df_final_1m['timestamp'].max()}")
print(f"Unique services: {df_final_1m['msname'].n_unique()}")


Loading Cleaned 1-Minute Data...
Loaded Shape: (937986, 8)
Columns: ['msname', 'timestamp', 'total_cpu_demand', 'total_memory_demand', 'total_traffic', 'avg_node_stress', 'avg_response_time', 'replica_count']
Timestamp range: 0 to 43140000
Unique services: 1303


## 2. Build Sliding Windows 

In [3]:
print("Building sliding windows...")

WINDOW_SIZE   = 5
PREDICT_STEPS = 1
FEATURES = ['total_traffic','avg_node_stress','replica_count','avg_response_time']

X_raw, y_raw = [], []
for name, group in df_final_1m.sort('timestamp').group_by('msname'):
    group   = group.with_columns(pl.col('avg_response_time').fill_null(0))
    X_array = group.select(FEATURES).to_numpy()
    y_array = group.select('total_cpu_demand').to_numpy()
    for i in range(WINDOW_SIZE, len(X_array) - PREDICT_STEPS + 1):
        X_raw.append(X_array[i-WINDOW_SIZE:i, :])
        y_raw.append(y_array[i:i+PREDICT_STEPS, 0])

X_raw = np.array(X_raw)
y_raw = np.array(y_raw).reshape(-1, PREDICT_STEPS)
print(f"X_raw Shape: {X_raw.shape}  ({WINDOW_SIZE} timesteps x {len(FEATURES)} features)")
print(f"y_raw Shape: {y_raw.shape}")
print(f"y_raw max:   {y_raw.max():.4f} cores | mean: {y_raw.mean():.4f} cores")


Building sliding windows...
X_raw Shape: (931471, 5, 4)  (5 timesteps x 4 features)
y_raw Shape: (931471, 1)
y_raw max:   1712.0461 cores | mean: 13.9350 cores


## 3. Split & Scale (identical to LSTM — 3D kept for GRU)

In [4]:
print("Splitting into Train/Validation sets...")
val_indices   = np.arange(0, len(X_raw), 4)
train_indices = np.setdiff1d(np.arange(len(X_raw)), val_indices)

X_train_raw = X_raw[train_indices];  X_val_raw = X_raw[val_indices]
y_train_raw = y_raw[train_indices];  y_val_raw = y_raw[val_indices]
print(f"X_train: {X_train_raw.shape} | X_val: {X_val_raw.shape}")
print(f"y_train: {y_train_raw.shape} | y_val: {y_val_raw.shape}")

print("\nScaling with Log Transform...")
n_train, timesteps, n_features = X_train_raw.shape

y_train_log = np.log1p(y_train_raw);  y_val_log = np.log1p(y_val_raw)
X_train_log = np.log1p(X_train_raw);  X_val_log = np.log1p(X_val_raw)

X_scaler = MinMaxScaler()
X_train_scaled = X_scaler.fit_transform(
    X_train_log.reshape(-1, n_features)).reshape(n_train, timesteps, n_features)
X_val_scaled   = X_scaler.transform(
    X_val_log.reshape(-1, n_features)).reshape(X_val_log.shape)

y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train_log)
y_val_scaled   = y_scaler.transform(y_val_log)

print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"y_train_scaled: {y_train_scaled.shape}")
print(f"y_scaler range: {y_scaler.data_min_[0]:.4f} to {y_scaler.data_max_[0]:.4f} (log scale)")


Splitting into Train/Validation sets...
X_train: (698603, 5, 4) | X_val: (232868, 5, 4)
y_train: (698603, 1) | y_val: (232868, 1)

Scaling with Log Transform...
X_train_scaled: (698603, 5, 4)
y_train_scaled: (698603, 1)
y_scaler range: 0.0000 to 7.4460 (log scale)


## 4. Build GRU Model

In [5]:
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("Building GRU Model...")
model = Sequential([
    GRU(64, input_shape=(WINDOW_SIZE, len(FEATURES)), return_sequences=True),
    Dropout(0.2),
    GRU(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(PREDICT_STEPS)
], name='GRU_model')
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
print(model.summary())


Building GRU Model...


Model: "GRU_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 5, 64)          │        13,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,393 (91.38 KB)

 Trainable params: 23,393 (91.38 KB)

 Non-trainable params: 0 (0.00 B)

None


## 5. Train

In [6]:
print("Starting Training...")
callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint("models/best_gru_model.keras", monitor='val_loss', save_best_only=True),
]
history = model.fit(
    X_train_scaled, y_train_scaled,
    epochs=60, batch_size=128,
    validation_data=(X_val_scaled, y_val_scaled),
    callbacks=callbacks, verbose=1
)
print("\nTraining Complete!")


Starting Training...
Epoch 1/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 41s 7ms/step - loss: 0.0023 - mae: 0.0343 - val_loss: 0.0018 - val_mae: 0.0307
Epoch 2/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 41s 7ms/step - loss: 0.0019 - mae: 0.0315 - val_loss: 0.0020 - val_mae: 0.0318
Epoch 3/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.0018 - mae: 0.0307 - val_loss: 0.0021 - val_mae: 0.0327
Epoch 4/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - loss: 0.0017 - mae: 0.0301 - val_loss: 0.0023 - val_mae: 0.0350
Epoch 5/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 42s 8ms/step - loss: 0.0017 - mae: 0.0296 - val_loss: 0.0020 - val_mae: 0.0341
Epoch 6/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - loss: 0.0016 - mae: 0.0292 - val_loss: 0.0020 - val_mae: 0.0344
Epoch 7/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 38s 7ms/step - loss: 0.0016 - mae: 0.0287 - val_loss: 0.0019 - val_mae: 0.0336
Epoch 8/60
5458/5458 ━━━━━━━━━━━━━━━━━━━━ 38s 7ms/step - loss: 0.0015 - mae: 0.0283 - val_loss: 0.0018 - val_mae: 0.0334
Epoch 9/60


## 6. Evaluate

In [7]:
print("Evaluating on full validation set...")
pred_scaled = model.predict(X_val_scaled)
# Inverse: unscale log -> expm1 -> CPU cores 
pred_actual = np.expm1(y_scaler.inverse_transform(pred_scaled.reshape(-1,1))).flatten()
true_actual = np.expm1(y_scaler.inverse_transform(y_val_scaled.reshape(-1,1))).flatten()

r2   = r2_score(true_actual, pred_actual)
mae  = mean_absolute_error(true_actual, pred_actual)
rmse = np.sqrt(mean_squared_error(true_actual, pred_actual))
mape = np.mean(np.abs((true_actual - pred_actual) / (true_actual + 1e-8))) * 100

print()
print('=' * 40)
print('MODEL EVALUATION REPORT')
print('=' * 40)
print(f'MAE  (Mean Absolute Error):     {mae:.4f} cores')
print(f'RMSE (Root Mean Squared Error): {rmse:.4f} cores')
print(f'R\u00b2   (Explained Variance):      {r2:.4f}')
print(f'MAPE (Mean Absolute % Error):   {mape:.2f}%')
print(f'Evaluated on:                   {len(true_actual):,} samples')
print('=' * 40)
save_result('GRU', r2, mae, rmse, mape)

Evaluating on full validation set...
7278/7278 ━━━━━━━━━━━━━━━━━━━━ 7s 887us/step

MODEL EVALUATION REPORT
MAE  (Mean Absolute Error):     3.5350 cores
RMSE (Root Mean Squared Error): 23.0419 cores
R²   (Explained Variance):      0.8344
MAPE (Mean Absolute % Error):   11374.22%
Evaluated on:                   232,868 samples
  Saved → results/all_models_results.csv


## 7. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'GRU — Training Loss, Prediction & Scatter  (R²={r2:.4f})', fontsize=13, fontweight='bold')

axes[0].plot(history.history['loss'],     color='#378ADD', label='Train')
axes[0].plot(history.history['val_loss'], color='#E24B4A', label='Val')
axes[0].set_title('Training & validation loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

n_show = min(200, len(true_actual))
axes[1].plot(true_actual[:n_show], color='#378ADD', linewidth=1.0, label='Actual CPU cores')
axes[1].plot(pred_actual[:n_show], color='#534AB7', linewidth=1.0, linestyle='--', label='GRU predicted')
axes[1].set_title(f'First {n_show} validation samples'); axes[1].set_ylabel('CPU cores'); axes[1].legend()

axes[2].scatter(true_actual, pred_actual, alpha=0.2, s=3, color='#534AB7')
lims = [min(true_actual.min(), pred_actual.min()), max(true_actual.max(), pred_actual.max())]
axes[2].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[2].set_title(f'Scatter  R²={r2:.4f}'); axes[2].set_xlabel('Actual (cores)'); axes[2].set_ylabel('Predicted'); axes[2].legend()

for ax in axes: ax.grid(True, alpha=0.25)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, '05_gru_results.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {out}")
print(f"Final — R²={r2:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f} | MAPE={mape:.2f}%")
